In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from transformers import pipeline
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings 
warnings.filterwarnings("ignore")

d:\Anaconda\envs\gensim-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

from sentence_transformers import SentenceTransformer

In [3]:
df = pd.read_csv("D:\Machine Learning\Datasets\Cases\indian_news_articles\historic_articles.csv")
df

,source,category,link,author,published_at,header,subheader,content
0,firstpost,india,https://www.firstpost.com/india/supreme-court-...,FP Staff,"September 21, 2020 20:11:56 IST","Supreme Court, in anti-CAA blockade case, says...",While listening to a batch of petitions agains...,The Supreme Court on Monday said there cannot ...
1,firstpost,india,https://www.firstpost.com/india/rti-reponse-re...,Sumit Chaturvedi,"September 20, 2020 10:02:10 IST",RTI reponse reveals over 18 lakh ration cards ...,The large backlog of pending ration card appli...,Lakhs of migrant workers who returned to their...
2,firstpost,india,https://www.firstpost.com/india/pilot-dies-as-...,Press Trust of India,"September 21, 2020 15:32:17 IST",Pilot dies as four-seater training aircraft cr...,"The aircraft, which belonged to the Indira Gan...",Azamgarh: A four-seater training aircraft cras...
3,firstpost,india,https://www.firstpost.com/india/hcl-technologi...,Press Trust of India,"September 21, 2020 14:15:16 IST",HCL Technologies to acquire Australia’s DWS Lt...,"DWS, with FY20 revenue at AUS $167.9 million, ...",New Delhi: HCL Technologies on Monday said it ...
4,firstpost,india,https://www.firstpost.com/india/parliament-mon...,FP Staff,"September 22, 2020 10:42:06 IST",Parliament LIVE Updates: Pralhad Joshi says go...,Parliament LIVE Updates: Parliamentary Affair ...,Parliament LATEST Updates: Parliamentary Affai...
...,...,...,...,...,...,...,...,...
1563,ndtv,world,https://www.ndtv.com/world-news/united-nations...,Reuters,"September 18, 2020 7:55 am IST",UN Asks Billionaires For Help As 270 Million P...,The combined wealth of America's billionaires ...,UN food chief David Beasley called on the worl...
1564,ndtv,world,https://www.ndtv.com/world-news/gunmen-kill-fi...,Agence France-Presse,"September 18, 2020 7:45 am IST",Gunmen Kill Five In Attack On Wake In Mexico,The assailants opened fire at a home in the ci...,Gunmen killed five people Thursday during an a...
1565,ndtv,world,https://www.ndtv.com/indians-abroad/us-congres...,Press Trust of India,"September 18, 2020 7:32 am IST",Deeply Concerned About India-China Border Issu...,"Over the past few months, top American lawmake...",An Indian-American Congressman has urged China...
1566,ndtv,world,https://www.ndtv.com/world-news/moderna-shares...,Agence France-Presse,"September 18, 2020 7:40 am IST","Moderna, Pfizer Share Covid Vaccine Trial Blue...","Pfizer, the other American company currently c...","US biotech firm Moderna, one of nine companies..."


In [24]:
df['header_info']=df['header'].fillna('')+df['subheader'].fillna('')
df['header_info'].isnull().sum()
df.columns
df['header_info']

0       Supreme Court, in anti-CAA blockade case, says...
1       RTI reponse reveals over 18 lakh ration cards ...
2       Pilot dies as four-seater training aircraft cr...
3       HCL Technologies to acquire Australia’s DWS Lt...
4       Parliament LIVE Updates: Pralhad Joshi says go...
                              ...                        
1563    UN Asks Billionaires For Help As 270 Million P...
1564    Gunmen Kill Five In Attack On Wake In MexicoTh...
1565    Deeply Concerned About India-China Border Issu...
1566    Moderna, Pfizer Share Covid Vaccine Trial Blue...
1567    Twitter Tightens Security For Politicians And ...
Name: header_info, Length: 1568, dtype: object

In [25]:
from bertopic import BERTopic
topic_model = BERTopic()
topics,probes=topic_model.fit_transform(df['header_info'].to_list())

In [26]:
topic_info=topic_model.get_topic_info()
list( topic_info[topic_info['Topic']==1]["Representative_Docs"])

[["France reports 10,569 new daily COVID-19 cases - World News , FirstpostPARIS (Reuters) - French health authorities reported 10,569 new confirmed COVID-19 cases on Sunday, down from the previous day's record increase of 13,498. The number of people in France who have died from COVID-19 infections has risen to 31,585 from the 31,274 published on Saturday, the health ministry said. The cumulative number of cases now stands at 453,763",
  "Second UK lockdown? PM says second wave inevitable, new restrictions possible - World News , FirstpostBy Guy Faulconbridge, Kate Holton and Andy Bruce LONDON (Reuters) - British Prime Minister Boris Johnson said he did not want another national lockdown but that new restrictions may be needed because the country was facing an 'inevitable' second wave of COVID-19. Ministers were on Friday reported to be considering a second national lockdown, after new COVID-19 cases almost doubled to 6,000 per day, hospital admissions rose and infection rates soared a

In [27]:
len(topics),df.shape

(1568, (1568, 10))

In [28]:
df['Topic']=topics
df=df.merge(topic_info, on='Topic',how='left')

In [29]:
df.head()

,source,category,link,author,published_at,header,subheader,content,header_info,topic_label,Topic,Count,Name,Representation,Representative_Docs
0,firstpost,india,https://www.firstpost.com/india/supreme-court-...,FP Staff,"September 21, 2020 20:11:56 IST","Supreme Court, in anti-CAA blockade case, says...",While listening to a batch of petitions agains...,The Supreme Court on Monday said there cannot ...,"Supreme Court, in anti-CAA blockade case, says...",2,3,79,3_court_delhi_case_the,"[court, delhi, case, the, india, police, of, i...",[Delhi Police arrests freelance journalist Raj...
1,firstpost,india,https://www.firstpost.com/india/rti-reponse-re...,Sumit Chaturvedi,"September 20, 2020 10:02:10 IST",RTI reponse reveals over 18 lakh ration cards ...,The large backlog of pending ration card appli...,Lakhs of migrant workers who returned to their...,RTI reponse reveals over 18 lakh ration cards ...,1,1,181,1_covid19_cases_the_coronavirus,"[covid19, cases, the, coronavirus, in, of, to,...","[France reports 10,569 new daily COVID-19 case..."
2,firstpost,india,https://www.firstpost.com/india/pilot-dies-as-...,Press Trust of India,"September 21, 2020 15:32:17 IST",Pilot dies as four-seater training aircraft cr...,"The aircraft, which belonged to the Indira Gan...",Azamgarh: A four-seater training aircraft cras...,Pilot dies as four-seater training aircraft cr...,-1,-1,188,-1_to_the_in_of,"[to, the, in, of, and, news, on, sports, for, by]",[Louisville to announce $12 million settlement...
3,firstpost,india,https://www.firstpost.com/india/hcl-technologi...,Press Trust of India,"September 21, 2020 14:15:16 IST",HCL Technologies to acquire Australia’s DWS Lt...,"DWS, with FY20 revenue at AUS $167.9 million, ...",New Delhi: HCL Technologies on Monday said it ...,HCL Technologies to acquire Australia’s DWS Lt...,-1,-1,188,-1_to_the_in_of,"[to, the, in, of, and, news, on, sports, for, by]",[Louisville to announce $12 million settlement...
4,firstpost,india,https://www.firstpost.com/india/parliament-mon...,FP Staff,"September 22, 2020 10:42:06 IST",Parliament LIVE Updates: Pralhad Joshi says go...,Parliament LIVE Updates: Parliamentary Affair ...,Parliament LATEST Updates: Parliamentary Affai...,Parliament LIVE Updates: Pralhad Joshi says go...,-1,2,95,2_sabha_bills_farm_rajya,"[sabha, bills, farm, rajya, opposition, bill, ...",[Parliament Passes Bill To Amend Companies Law...


In [31]:
import ipywidgets as widgets

In [ ]:
def retrive_docs(topic_id):
    return df[df['name']==topic_id]['header_info']
widgets.interact(retrive_docs, topic_id=list(topic_info['Name'].unique()))

ValueError: cannot find widget or abbreviation for argument: 'topic_id'

In [23]:
retreive_docs(4)

536    Premier League: Manchester City begin season w...
537    Premier League: Arsenal sign Dijon goalkeeper ...
542    Premier League: Brazilian defender Thiago Silv...
560    Premier League: Leicester City ready to 'attac...
564    Premier League: Liverpool's Alisson shows Chel...
571    Premier League: Pep Guardiola says he 'has to ...
574    Premier League: Sadio Mane brace sees Liverpoo...
577    Premier League: Dele Alli's Tottenham future i...
578    Premier League: Donny van de Beek slams 'slow'...
586    Bundesliga: Erling Haaland double inspires Bor...
588    Premier League: Gareth Bale returns to Tottenh...
593    Premier League: Manchester City's Sergio Aguer...
595    Premier League: Wilfried Zaha nets brace as Cr...
596    Premier League: Liverpool sign Portuguese wing...
601    Premier League: Aston Villa sign Burkina Faso ...
603    Premier League: Manchester United's Ole Gunnar...
604    Premier League: Liverpool face Chelsea test as...
609    Liverpool manager Jurgen